# Connect to google account

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Global Variables

In [ ]:
import easydict
args = easydict.EasyDict()

# file path
args.default_path = "/content/drive/MyDrive/Study/kaggle_competition/data/" # 파일이 존재하는 폴더
args.train_csv = args.default_path + "train.csv" # 학습용 데이터셋
args.test_csv = args.default_path + "test.csv" # 제출용 예측 데이터셋
args.default_submission_csv = args.default_path + "submission.csv" # 제출에 사용할 파일
args.submission_csv = "model_with_cv_0820_trial02.csv" # 제출용 파일


In [ ]:
args.train_csv == "/content/drive/MyDrive/Study/kaggle_competition/data/train.csv"

True

# reset_seeds

In [ ]:
import os
import numpy as np
import random
import torch

def reset_seeds(seed=42):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)    # 파이썬 환경변수 시드 고정
  np.random.seed(seed)
  torch.manual_seed(seed) # cpu 연산 무작위 고정
  torch.cuda.manual_seed(seed) # gpu 연산 무작위 고정
  torch.backends.cudnn.deterministic = True  # cuda 라이브러리에서 Deterministic(결정론적)으로 예측하기 (예측에 대한 불확실성 제거 )

# Load Dataset

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
ori_train = pd.read_csv(args.train_csv)
ori_test = pd.read_csv(args.test_csv)

ori_train.shape, ori_test.shape

((916, 12), (393, 11))

# EDA

In [ ]:
ori_train.head()

,passengerid,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,0,2,"Wheeler, Mr. Edwin Frederick""""",male,NaN,0,0,SC/PARIS 2159,12.8750,NaN,S
1,1,0,3,"Henry, Miss. Delia",female,NaN,0,0,382649,7.7500,NaN,Q
2,2,1,1,"Hays, Mrs. Charles Melville (Clara Jennings Gr...",female,52.0,1,1,12749,93.5000,B69,S
3,3,1,3,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.0,0,0,350043,7.7958,NaN,S
4,4,0,2,"Hold, Mr. Stephen",male,44.0,1,0,26707,26.0000,NaN,S


# Cleaning Dataset

In [ ]:
ori_train.shape, ori_test.shape

((916, 12), (393, 11))

In [ ]:
# 필요없는 컬럼 제거
ori_train.drop(['passengerid'], axis=1, inplace=True)
ori_test.drop(['passengerid'], axis=1, inplace=True)

ori_train.shape, ori_test.shape

((916, 11), (393, 10))

In [ ]:
# drop -> 삭제하다.
# duplicates -> row 데이터에 대한 중복데이터
# drop_duplicates -> row 데이터들 중 중복 데이터 삭제
ori_train.drop_duplicates(inplace=True, keep='first')
ori_test.drop_duplicates(inplace=True, keep='first')

ori_train.shape, ori_test.shape

((916, 11), (393, 10))

## 결측치 제거

In [ ]:
(ori_train.isnull().sum() / len(ori_train)).sort_values(ascending=False)

,0
cabin,0.783843
age,0.196507
embarked,0.001092
name,0.000000
pclass,0.000000
survived,0.000000
gender,0.000000
parch,0.000000
sibsp,0.000000
fare,0.000000


In [ ]:
ori_train_none_cols = ori_train.isnull().sum()[ori_train.isnull().sum() > 0].index
ori_train_none_cols

Index(['age', 'cabin', 'embarked'], dtype='object')

In [ ]:
ori_test_none_cols = ori_test.isnull().sum()[ori_test.isnull().sum() > 0].index
ori_test_none_cols

Index(['age', 'fare', 'cabin', 'embarked'], dtype='object')

In [ ]:
none_cols = list(set(ori_train_none_cols) | set(ori_test_none_cols))
none_cols # 결측치 컬럼

['cabin', 'fare', 'embarked', 'age']

In [ ]:
for col in none_cols:
  try:
    # 통계 값 추출
    _value = ori_train[col].mean() # 수치형 데이터
  except:
    _value = ori_train[col].mode()[0] # 범주형 데이터
  finally:
    # 결측치에 통계값 넣기
    ori_train[col].fillna(_value, inplace=True)
    ori_test[col].fillna(_value, inplace=True)

/tmp/ipython-input-787063171.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ori_train[col].fillna(_value, inplace=True)
/tmp/ipython-input-787063171.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [ ]:
ori_train.isnull().sum().sum(), ori_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

# 인코딩 처리

In [ ]:
ori_train.columns

Index(['survived', 'pclass', 'name', 'gender', 'age', 'sibsp', 'parch',
       'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')

In [ ]:
drop_cols = [
    'name', 'ticket'
]
ori_train.drop(drop_cols, axis=1, inplace=True)
ori_test.drop(drop_cols, axis=1, inplace=True)

In [ ]:
!pip install category_encoders

In [ ]:
# !pip install category_encoders
import category_encoders as ce

reset_seeds()
encoding_cols = [
    'pclass', 'gender', 'cabin', 'embarked'
]

encoder = ce.OneHotEncoder(use_cat_names=True) # 인코더 생성
encoder.fit(ori_train[encoding_cols]) # 범주형 데이터 변환 학습

# 변환
train_one_hot = encoder.transform(ori_train[encoding_cols])
test_one_hot = encoder.transform(ori_test[encoding_cols])

In [ ]:
ori_train[encoding_cols].shape, train_one_hot.shape

((916, 4), (916, 152))

In [ ]:
ori_test[encoding_cols].shape, test_one_hot.shape

((393, 4), (393, 152))

In [ ]:
# none_encoding_cols -> ori_test용
# none_encoding_cols + survived -> ori_train용
# survived
none_encoding_cols = list(set(ori_test.columns) - set(encoding_cols))

# ori_train
train = pd.concat([
    ori_train[none_encoding_cols + ['survived']], # 수치형 데이터
    train_one_hot # 범주형 데이터
    ], axis=1)

# ori_test
test = pd.concat([
    ori_test[none_encoding_cols], # 수치형 데이터
    test_one_hot # 범주형 데이터
    ], axis=1)

print(f"before: {ori_train.shape} / {ori_test.shape}")
print(f"after: {train.shape} / {test.shape}")

before: (916, 9) / (393, 8)
after: (916, 157) / (393, 156)


# 데이터 증강

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
# 클래스별 비율 확인
train['survived'].value_counts() / len(train)

,count
survived,
0,0.622271
1,0.377729


In [ ]:
train.shape # features & targets

(916, 157)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Columns: 157 entries, sibsp to embarked_C
dtypes: float64(2), int64(155)
memory usage: 1.1 MB


In [ ]:
train.shape

(916, 157)

In [ ]:
train.select_dtypes(
    exclude=np.number
).shape # 데이터값이 0인지 확인하자!!

(916, 0)

In [ ]:
reset_seeds()

smote = SMOTE()
smote_features, smote_targets = smote.fit_resample(train.drop(['survived'], axis=1), train['survived'])

smote_features.shape, smote_targets.shape

((1140, 156), (1140,))

In [ ]:
smote_targets.value_counts() / len(smote_targets)

,count
survived,
0,0.5
1,0.5


# 모델링

In [ ]:
smote_features.isnull().sum().sum(), test.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [ ]:
smote_features.shape, test.shape

((1140, 156), (393, 156))

In [ ]:
len(smote_features) == len(smote_targets)

True

## 모델 생성

In [ ]:
from lightgbm import LGBMClassifier, plot_importance

In [ ]:
reset_seeds()

model = LGBMClassifier(verbose=-1)

## CV

In [ ]:
from sklearn.model_selection import StratifiedKFold

reset_seeds()

cv = StratifiedKFold(n_splits=5, shuffle=True)

In [ ]:
from sklearn.metrics import roc_auc_score
reset_seeds()

# len(smote_features) == len(smote_targets)
for i, (train_index, valid_index) in enumerate(cv.split(smote_features, smote_targets)):
  # 학습용 데이터 -> features, targets
  tr_features, tr_targets = smote_features.iloc[train_index], smote_targets.iloc[train_index]
  # 평가용 데이터 -> features, targests
  te_features, te_targets = smote_features.iloc[valid_index], smote_targets.iloc[valid_index]

  # 모델 학습
  model.fit(tr_features, tr_targets)

  # 학습용 점수
  tr_predictions = model.predict_proba(tr_features)[:, 1]
  tr_score = roc_auc_score(tr_targets, tr_predictions)
  # 평가용 점수
  te_predictions = model.predict_proba(te_features)[:, 1]
  te_score = roc_auc_score(te_targets, te_predictions)

  print(f"{i+1}번째 점수오차: {te_score - tr_score} / 학습용 점수: {tr_score} / 평가용 점수: {te_score}")

1번째 점수는 0.9259002770083103
2번째 점수는 0.935518621114189
3번째 점수는 0.9005847953216374
4번째 점수는 0.9564481378885812
5번째 점수는 0.9317482302246846


# 제출용 만들자

In [ ]:
reset_seeds()
pred = model.predict_proba(test)

len(pred) == len(test)

True